In [5]:
import json
from pathlib import Path
from statistics import mean

# Search scope requested by user
DATASETS = ["lvb", "mlvu", "videomme"]

base = Path("../../results/vseek")


def load_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


# Build lookup for VSeek-Em and VSeek-Puls logs (restricted to lvb/mlvu/videomme)
em_logs = {}
puls_logs = {}
for ds in DATASETS:
    em_logs[ds] = load_jsonl(base / "vseek" / ds / "agent_results.jsonl")
    puls_logs[ds] = load_jsonl(base / "vseek-puls" / ds / "agent_results.jsonl")
    print(len(em_logs[ds]), len(puls_logs[ds]))

# print("Length of em_logs:", len(em_logs["lvb"]), len(em_logs["mlvu"]), len(em_logs["videomme"]))
# print("Length of puls_logs:", len(puls_logs["lvb"]), len(puls_logs["mlvu"]), len(puls_logs["videomme"]))

# Candidate condition:
# 1) VSeek-Puls majority answer is correct
# 2) at least 6 QA turns in that trial (max(turns) >= 6)
# 3) prefer longer-video dataset -> prioritize lvb
candidates = []
for ds in DATASETS:
    em_index = {}
    for r in em_logs.get(ds, []):
        em_index[r["video_id"]] = [r] if r["video_id"] not in em_index else em_index[r["video_id"]] + [r]
        #em_index[r["video_id"]] = [r]
    # Filter out those where both are incorrect
    em_filtered = {}
    for key, r in em_index.items():
        any_correct = any([r["is_majority_correct"] for r in r])
        if not any_correct:  
            if len(r) == 1:
                em_filtered[r[0]["video_id"]] = r[0]
    
    print(len(em_filtered), "candidates from VSeek-EM", ds)        
    for r in puls_logs.get(ds, []):
        if not r.get("is_majority_correct", False):
            continue
        
        if r["video_id"] not in em_filtered:
            continue
        
        turns = [int(t) for t in r.get("turns", []) if str(t).isdigit()]
        if not turns or max(turns) < 6:
            continue
        vid = r["video_id"]
        candidates.append(
            {
                "dataset": ds,
                "video_id": vid,
                "puls": r,
                "em": em_index.get(vid),
                "max_turn": max(turns),
                "avg_turn": round(mean(turns), 2),
                "qid": r.get("qid"),
            }
        )

# Sort by user preference: longer-video dataset first (lvb), then richer turn usage
dataset_priority = {"lvb": 0, "videomme": 1, "mlvu": 2}
candidates = sorted(
    candidates,
    key=lambda x: (
        dataset_priority.get(x["dataset"], 99),
        -x["max_turn"],
        -x["avg_turn"],
        -x["puls"].get("correct_count", 0),
    ),
)

selected = candidates[0] if candidates else None
print("Total candidates:", len(candidates))
print("Selected example:", selected["dataset"], selected["video_id"] if selected else None)



1337 1337
2174 2174
2700 2700
106 candidates from VSeek-EM lvb
315 candidates from VSeek-EM mlvu
0 candidates from VSeek-EM videomme
Total candidates: 141
Selected example: lvb GKgl3aJr5Xw


In [6]:
# Compiled entry to keep in notebook

# I now want to check the length of the video corresponding to the selected example
# Use the same source as plot_pass_k.ipynb for LVB durations.
LVB_DATASET_ROOT = Path("/nas/mars/dataset/longvideobench/LongVideoBench")
MLVU_DATASET_ROOT = Path("/nas/mars/dataset/MLVU/MLVU")

def to_seconds_from_value(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        return None


def extract_duration_from_original_data(item: dict):
    for k in ["duration", "video_duration", "duration_sec", "duration_seconds", "length", "video_length", "seconds"]:
        if k in item:
            sec = to_seconds_from_value(item.get(k))
            if sec is not None:
                return sec
    return None


def build_lvb_duration_map(dataset_root: Path):
    out = {}
    lvb_file = dataset_root / "puls_refined.json"
    if not lvb_file.exists():
        return out

    with lvb_file.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        return out

    for item in data:
        meta = item.get("metadata", {})
        vid = meta.get("video_id")
        original_data = meta.get("original_data")
        if not vid or not original_data:
            continue
        try:
            original = json.loads(original_data)
        except json.JSONDecodeError:
            continue
        sec = extract_duration_from_original_data(original)
        if sec is not None and vid not in out:
            out[vid] = sec

    return out



lvb_duration_map = build_lvb_duration_map(LVB_DATASET_ROOT)
mlvu_duration_map = build_lvb_duration_map(MLVU_DATASET_ROOT)

for j, c in enumerate(candidates):
    if c["dataset"] == "lvb":
        duration_sec = lvb_duration_map.get(c["video_id"])
    elif c["dataset"] == "mlvu":
        duration_sec = mlvu_duration_map.get(c["video_id"])
    if duration_sec is not None:
        print(f"Video duration for {c['video_id']}: {duration_sec:.2f}s ({duration_sec/60:.2f} min)")
        candidates[j]["video_duration_sec"] = duration_sec
        candidates[j]["video_duration_min"] = round(duration_sec / 60, 2)
    else:
        print(f"Could not find duration for {c['video_id']} in {LVB_DATASET_ROOT / 'puls_refined.json'}")


# Build a full candidate list with corresponding question(s).
DATASET_ROOTS = {
    "lvb": Path("/nas/mars/dataset/longvideobench/LongVideoBench"),
    "mlvu": Path("/nas/mars/dataset/MLVU/MLVU"),
    "videomme": Path("/nas/mars/dataset/Video-MME"),
}


def extract_question_from_item(item: dict):
    q = item.get("question")
    if q:
        return str(q)

    meta = item.get("metadata", {})
    raw = meta.get("original_data")
    if raw:
        try:
            original = json.loads(raw)
            for key in ["question", "query", "prompt"]:
                if key in original and original[key]:
                    return str(original[key])
        except json.JSONDecodeError:
            return None
    return None


def build_question_map_from_puls_refined(dataset_root: Path):
    out = {}
    src = dataset_root / "puls_refined.json"
    if not src.exists():
        return out

    with src.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        return out

    for item in data:
        meta = item.get("metadata", {})
        vid = meta.get("video_id")
        if not vid:
            continue
        q = extract_question_from_item(item)
        if not q:
            continue
        if vid not in out:
            out[vid] = []
        if q not in out[vid]:
            out[vid].append(q)

    return out

def build_data(dataset_root: Path):
    out = {}
    src = dataset_root / "puls_refined.json"
    if not src.exists():
        return out

    with src.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        return out

    return data
# question_maps = {
#     ds: build_question_map_from_puls_refined(DATASET_ROOTS[ds])
#     for ds in DATASETS
# }

# for ds, qs in question_maps.items():
#     print(f"{ds}: {len(qs)}")

# total_questions = 0
# candidates_with_questions = []
# for j, c in enumerate(candidates):
#     ds = c["dataset"]
#     vid = c["video_id"]
#     questions = question_maps.get(ds, {}).get(vid, [])
    
#     candidates_with_questions.append(
#         {
#             "dataset": ds,
#             "video_id": vid,
#             "max_turn": c["max_turn"],
#             "avg_turn": c["avg_turn"],
#             "puls_correct_count": c["puls"].get("correct_count", 0),
#             "num_questions": len(questions),
#             "questions": questions,
#         }
#     )
#     total_questions += len(questions)
#     # merge dicts
#     candidates[j] = {**candidates[j], **candidates_with_questions[j]}
#     #print(candidates[j])
# print(total_questions)

# compiled_entry["all_candidates_with_questions"] = candidates_with_questions
# print("Candidates with questions:", len(candidates_with_questions))

# compiled_entry

candidates_with_questions = []
question_maps = {ds: build_data(DATASET_ROOTS[ds]) for ds in DATASETS}
total_questions = 0

for j,c in enumerate(candidates):
    qid = c["qid"]
    if question_maps[c["dataset"]][qid]['metadata']['video_id'] == c["video_id"]:
        candidates_with_questions.append(
            {
                "dataset": c["dataset"],
                "video_id": c["video_id"],
                "max_turn": c["max_turn"],
                "avg_turn": c["avg_turn"],
                "puls_correct_count": c["puls"].get("correct_count", 0),
                "num_questions": len(question_maps[c["dataset"]][qid]['question']),
                "questions": question_maps[c["dataset"]][qid]['question'],
            }
        )
        total_questions += len(question_maps[c["dataset"]][qid]['question'])
        candidates[j] = {**candidates[j], **candidates_with_questions[-1]}
print(total_questions)
    
    

Video duration for GKgl3aJr5Xw: 13.00s (0.22 min)
Video duration for @kelseyinlondon-7258968758130085146: 11.50s (0.19 min)
Video duration for @healthfood-6948509102305791238: 47.33s (0.79 min)
Video duration for @luxtravelbe-7286546546932370721: 27.03s (0.45 min)
Video duration for @movie.explained6-7269746510462536962: 58.70s (0.98 min)
Video duration for @placesunleashed-7326709884102216965: 17.40s (0.29 min)
Video duration for T1K4rgs-1b8: 12.00s (0.20 min)
Video duration for @lisolna-7282789187676294432: 60.37s (1.01 min)
Video duration for GZc0P3Apfx4: 49.02s (0.82 min)
Video duration for 14ot4DrXdds: 8.01s (0.13 min)
Video duration for @placesunleashed-7321079612488862981: 20.07s (0.33 min)
Video duration for @lisolna-7321010343067618592: 60.63s (1.01 min)
Video duration for kFHVBCEwC3w: 13.00s (0.22 min)
Video duration for H_b5d-rLXJU: 10.01s (0.17 min)
Video duration for TARe4G-SXfk: 13.97s (0.23 min)
Video duration for 6Z7AAcD8rbo: 13.01s (0.22 min)
Video duration for lNReCCS

In [14]:
import sys
import json
import random
from pathlib import Path

import cv2
import datasets

# Make src imports available when running from notebooks/plots
PROJECT_ROOT = Path("../../").resolve()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from data.prompts.prompts import tagbased, openaitooluse, tagbasedsummary, fanout
from vseek.data.frame import VideoFrames

# -----------------------------
# Config
# -----------------------------
PROMPT_TYPE = "tag"            # tag | openai | tagsummary | fanout
TRAIN_RATIO = 0.0               # 0.0 => all rows go to test split (inference only)
SEED = 42
WRITE_SPLITS = True             # writes train.parquet + test.parquet

# Frame packaging config (matches preprocessor behavior)
EMBED_FRAMES = True
WINDOW_SIZE = 8
MAX_FRAMES_PER_TURN = 16
THUMB_MAX_SIDE = 224
THUMB_QUALITY = 85

INDEX_ROOTS = {
    # each path should contain <dataset>_window_<WINDOW_SIZE>/<video_id>/
    "lvb": Path("/home/hg22723/vseek/dataset"),
    "mlvu": Path("/home/hg22723/vseek/dataset"),
    "videomme": Path("/home/hg22723/vseek/dataset"),
}

OUTPUT_DIR = PROJECT_ROOT / "mined_payloads" / "puls_refined_selective"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PULS_REFINED_FILES = {
    "lvb": Path("/nas/mars/dataset/longvideobench/LongVideoBench/puls_refined.json"),
    "mlvu": Path("/nas/mars/dataset/MLVU/MLVU/puls_refined.json"),
    "videomme": Path("/nas/mars/dataset/Video-MME/puls_refined.json"),
}

# -----------------------------
# Helpers
# -----------------------------
def _pick_system_prompt(prompt_type: str) -> str:
    if prompt_type == "tag":
        return tagbased.system_prompt
    if prompt_type == "openai":
        return openaitooluse.system_prompt
    if prompt_type == "tagsummary":
        return tagbasedsummary.system_prompt
    if prompt_type == "fanout":
        return fanout.system_prompt
    raise ValueError(f"Invalid prompt type: {prompt_type}")


def build_prompt(prompt_type: str, entry: dict) -> list[dict]:
    question_text = str(entry.get("question", "")).strip()
    candidates = entry.get("candidates", []) or []
    options_block = "".join([f"\\n{i}) {opt}" for i, opt in enumerate(candidates)]) if candidates else ""
    user_content = question_text + "\\nOptions: \\n" + options_block

    return [
        {"role": "system", "content": _pick_system_prompt(prompt_type)},
        {"role": "user", "content": user_content},
    ]


def normalize_q(text: str) -> str:
    return " ".join(str(text).split()).strip().lower()


def _encode_frame(frame, max_side: int, quality: int) -> bytes:
    h, w = frame.shape[:2]
    scale = max_side / max(h, w) if max(h, w) > max_side else 1.0
    if scale < 1.0:
        frame = cv2.resize(
            frame,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    ok, buffer = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    if not ok:
        raise ValueError("Could not encode frame")
    return buffer.tobytes()


def _load_encoded_frames_for_video(dataset: str, video_id: str):
    if not EMBED_FRAMES:
        return None, None
    index_root = INDEX_ROOTS.get(dataset)
    if index_root is None:
        return None, None

    vf_dir = index_root / f"{dataset}_window_{WINDOW_SIZE}" / str(video_id)
    if not vf_dir.exists():
        return None, None

    try:
        video_frames = VideoFrames.load(str(vf_dir))

        frames_by_window = []
        for window_idx in video_frames.frames_by_window.keys():
            chunk = video_frames.get_frame_chunk(window_idx)
            encoded = [_encode_frame(f, THUMB_MAX_SIDE, THUMB_QUALITY) for f in chunk]
            frames_by_window.append({
                "window_idx": window_idx,
                "encoded_frames": encoded,
            })

        video_summary = video_frames.uniformly_sample_frames(MAX_FRAMES_PER_TURN)
        encoded_video_summary = [_encode_frame(f, THUMB_MAX_SIDE, THUMB_QUALITY) for f in video_summary]
        return frames_by_window, encoded_video_summary
    except Exception as e:
        print(f"[warn] frame packaging failed for {dataset}/{video_id}: {e}")
        return None, None


# -----------------------------
# Build mined video/question lookup from notebook candidates
# -----------------------------
# Expects `candidates` built in previous cells
mined_lookup = {}
for c in candidates:
    ds = c["dataset"]
    vid = c["video_id"]
    qs = c.get("questions", []) or []
    mined_lookup[(ds, vid)] = {
        "candidate": c,
        "question_set": set(normalize_q(q) for q in qs if q),
    }

print(f"Mined video candidates: {len(mined_lookup)}")


# -----------------------------
# Load puls_refined rows and filter to mined subset
# -----------------------------
rows = []
row_idx = 0

for ds, path in PULS_REFINED_FILES.items():
    if not path.exists():
        print(f"[skip] missing {path}")
        continue

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    kept_for_ds = 0
    for item in data:
        meta = item.get("metadata", {}) or {}
        vid = str(meta.get("video_id", ""))
        key = (ds, vid)
        if key not in mined_lookup:
            continue

        # Optional question-level filtering if notebook mined explicit questions for that video
        qset = mined_lookup[key]["question_set"]
        q_now = str(item.get("question", ""))
        if qset and normalize_q(q_now) not in qset:
            continue

        messages = build_prompt(PROMPT_TYPE, item)
        correct_choice = item.get("correct_choice", None)
        cmeta = mined_lookup[key]["candidate"]

        execute_kwargs = {
            "topk": 4,
            "video_id": vid,
            "dataset": ds,
            "puls": item.get("puls", {}),
        }

        frames_by_window, encoded_video_summary = _load_encoded_frames_for_video(ds, vid)
        if frames_by_window is not None and encoded_video_summary is not None:
            execute_kwargs["precomputed_frames"] = frames_by_window
            execute_kwargs["video_summary"] = encoded_video_summary

        rows.append(
            {
                "data_source": ds,
                "prompt": messages,
                "ability": "video_reasoning",
                "reward_model": {
                    "style": "exact_match",
                    "ground_truth": str(correct_choice) if correct_choice is not None else None,
                },
                "extra_info": {
                    "index": row_idx,
                    "question": item.get("question", ""),
                    "candidates": item.get("candidates", []),
                    "correct_choice": str(correct_choice) if correct_choice is not None else None,
                    "metadata": meta,
                    "paths": {
                        "raw_video_path": item.get("paths", {}).get("raw_video_path"),
                        "subtitle_path": item.get("paths", {}).get("subtitle_path"),
                        "video_path": item.get("paths", {}).get("video_path"),
                    },
                    "selection_meta": {
                        "selection_source": "mine_examples_notebook",
                        "video_id": cmeta.get("video_id"),
                        "max_turn": cmeta.get("max_turn"),
                        "avg_turn": cmeta.get("avg_turn"),
                        "puls_correct_count": cmeta.get("puls", {}).get("correct_count"),
                    },
                    "tools_kwargs": {
                        "video_search": {
                            "execute_kwargs": execute_kwargs
                        }
                    },
                },
            }
        )
        row_idx += 1
        kept_for_ds += 1

    print(f"[{ds}] kept rows: {kept_for_ds}")

print(f"Total rows kept: {len(rows)}")
if not rows:
    raise RuntimeError("No rows were selected. Check candidate/video/question matching.")


# -----------------------------
# Write inference parquet (single file)
# -----------------------------
inference_path = OUTPUT_DIR / f"inference_{PROMPT_TYPE}.parquet"
ds_all = datasets.Dataset.from_list(rows)
ds_all.to_parquet(str(inference_path))
print(f"Wrote inference parquet: {inference_path}")


# -----------------------------
# Optional: write train/test split parquet
# -----------------------------
if WRITE_SPLITS:
    random.seed(SEED)
    idxs = list(range(len(rows)))
    random.shuffle(idxs)
    split = int(len(idxs) * TRAIN_RATIO)
    train_ids = set(idxs[:split])

    train_rows, test_rows = [], []
    for i, r in enumerate(rows):
        rr = dict(r)
        rr["extra_info"] = dict(r["extra_info"])
        if i in train_ids:
            rr["extra_info"]["split"] = "train"
            train_rows.append(rr)
        else:
            rr["extra_info"]["split"] = "test"
            test_rows.append(rr)

    train_path = OUTPUT_DIR / f"train_{PROMPT_TYPE}.parquet"
    test_path = OUTPUT_DIR / f"test_{PROMPT_TYPE}.parquet"

    datasets.Dataset.from_list(train_rows if train_rows else []).to_parquet(str(train_path))
    datasets.Dataset.from_list(test_rows if test_rows else []).to_parquet(str(test_path))

    print(f"Wrote train parquet: {train_path} ({len(train_rows)} rows)")
    print(f"Wrote test parquet:  {test_path} ({len(test_rows)} rows)")


# -----------------------------
# Write model-ready payload JSONL for selective inference
# -----------------------------
payload_path = OUTPUT_DIR / f"payload_{PROMPT_TYPE}.jsonl"
with payload_path.open("w", encoding="utf-8") as f:
    for r in rows:
        payload = {
            "messages": r["prompt"],
            "tools_kwargs": r["extra_info"]["tools_kwargs"],
            "gt": r["reward_model"]["ground_truth"],
            "video_id": r["extra_info"]["metadata"].get("video_id"),
            "dataset": r["data_source"],
            "question": r["extra_info"].get("question", ""),
        }
        # f.write(json.dumps(payload, ensure_ascii=False) + "\\n")

print(f"Wrote payload jsonl: {payload_path}")
print("Done.")

Mined video candidates: 141
[lvb] kept rows: 23


KeyboardInterrupt: 

In [27]:
# From the mined responses, first obtain pred text + video_id for VSeek-EM and VSeek-Puls (mined)
import json
from pathlib import Path

EM_PATH = Path("../../results/vseek/vseek/mined/agent_results.jsonl")
PULS_PATH = Path("../../results/vseek/vseek-puls/mined/agent_results.jsonl")


def load_jsonl_as_qid_map(path: Path):
    out = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            out[row["qid"]] = row
    return out


def pred_text_for_majority(row: dict):
    """
    Return the pred text corresponding to row['majority_vote'].
    Fallback to pred[0] if no parsed_pred match.
    """
    preds = row.get("pred", []) or []
    parsed = row.get("parsed_pred", []) or []
    majority = str(row.get("majority_vote", ""))

    idx = 0
    for i, p in enumerate(parsed):
        if str(p) == majority:
            idx = i
            break

    text = preds[idx] if idx < len(preds) else (preds[0] if preds else "")
    return " ".join(str(text).replace("<|endoftext|>", " ").split())


def max_turn(row: dict):
    turns = [int(t) for t in row.get("turns", []) if str(t).isdigit()]
    return max(turns) if turns else -1


def summarize_case(qid: int, em_row: dict, puls_row: dict):
    print(puls_row.get("question"))
    print(em_row.get("question"))
    assert puls_row.get("question") == em_row.get("question")
    return {
        "qid": qid,
        "question": puls_row.get("question"),
        "candidates": puls_row.get("candidates"),
        "video_id": puls_row.get("video_id") or em_row.get("video_id"),
        "gt": puls_row.get("gt", em_row.get("gt")),
        "puls": {
            "is_majority_correct": bool(puls_row.get("is_majority_correct", False)),
            "majority_vote": str(puls_row.get("majority_vote")),
            "max_turn": max_turn(puls_row),
            "pred_text": pred_text_for_majority(puls_row),
        },
        "em": {
            "is_majority_correct": bool(em_row.get("is_majority_correct", False)),
            "majority_vote": str(em_row.get("majority_vote")),
            "max_turn": max_turn(em_row),
            "pred_text": pred_text_for_majority(em_row),
        },
    }


em_map = load_jsonl_as_qid_map(EM_PATH)
puls_map = load_jsonl_as_qid_map(PULS_PATH)
common_qids = sorted(set(em_map).intersection(puls_map))

print(f"Loaded qids: EM={len(em_map)} Puls={len(puls_map)} Common={len(common_qids)}")

# Constraint requested by user: ensure VSeek-Puls uses > 6 turns
puls_max_turns = [max_turn(puls_map[q]) for q in common_qids]
num_gt6 = sum(t > 6 for t in puls_max_turns)
print(f"VSeek-Puls turn check: max_turn_overall={max(puls_max_turns)} | num_with_turn_gt_6={num_gt6}")

all_vseek = [
]

for q in common_qids:
    all_vseek.append(summarize_case(q, em_map[q], puls_map[q]))

# 1) Two examples where VSeek (Puls) gets right and EM gets wrong
vseek_right_em_wrong = []
for q in common_qids:
    p = puls_map[q]
    e = em_map[q]
    if bool(p.get("is_majority_correct", False)) and not bool(e.get("is_majority_correct", False)):
        vseek_right_em_wrong.append(summarize_case(q, e, p))

# 2) Two examples where EM variant gets wrong (any, regardless of Puls correctness)
em_wrong = []
for q in common_qids:
    e = em_map[q]
    if not bool(e.get("is_majority_correct", False)):
        em_wrong.append(summarize_case(q, e, puls_map[q]))

print(f"VSeek right + EM wrong count: {len(vseek_right_em_wrong)}")
print(f"EM wrong count: {len(em_wrong)}")

picked_vseek = vseek_right_em_wrong
picked_em_wrong = em_wrong[:2]


def print_examples(title: str, items: list[dict]):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    for i, ex in enumerate(items, 1):
        print(f"[{i}] qid={ex['qid']} | video_id={ex['video_id']} | gt={ex['gt']}")
        print(f"Question: {ex['question']}")
        # print(f"Answer: {ex['answer']}")
        print(
            f"  Puls: correct={ex['puls']['is_majority_correct']} "
            f"vote={ex['puls']['majority_vote']} max_turn={ex['puls']['max_turn']}"
        )
        print(
            f"  EM:   correct={ex['em']['is_majority_correct']} "
            f"vote={ex['em']['majority_vote']} max_turn={ex['em']['max_turn']}"
        )
        print(f"  Puls pred_text: {ex['puls']['pred_text'][:280]}")
        print(f"  EM   pred_text: {ex['em']['pred_text'][:280]}")
        print("-" * 100)


print_examples("Two examples: VSeek-Puls correct while EM is wrong", picked_vseek)
# print_examples("Two examples: EM variant wrong", picked_em_wrong)

# Optional: if you strictly need Puls max_turn > 6 for picked cases, this will be empty for current mined set
picked_vseek_gt6 = [x for x in vseek_right_em_wrong if x["puls"]["max_turn"] > 6][:2]
print("\nStrict filter (Puls max_turn > 6) examples found:", len(picked_vseek_gt6))
if not picked_vseek_gt6:
    print("No mined examples satisfy Puls max_turn > 6 in this file.")

Loaded qids: EM=140 Puls=140 Common=140
VSeek-Puls turn check: max_turn_overall=6 | num_with_turn_gt_6=0

 This is a multiple choice question. You must choose the correct answer as a number. 
Question: Is there any abnormality in this surveillance video? If so, what type of abnormality is it? 


 This is a multiple choice question. You must choose the correct answer as a number. 
Question: Is there any abnormality in this surveillance video? If so, what type of abnormality is it? 


 This is a multiple choice question. You must choose the correct answer as a number. 
Question: Are there any irregularities in this surveillance video? If there are, what sort are they? 


 This is a multiple choice question. You must choose the correct answer as a number. 
Question: Are there any irregularities in this surveillance video? If there are, what sort are they? 


 This is a multiple choice question. You must choose the correct answer as a number. 
Question: What name was written on the door? 


In [34]:
# Split pred_text into turn-level events: each turn is either `images` or `answer`
import re


def split_pred_text_per_turn(pred_text: str):
    """
    Parse a raw model trajectory string into ordered turns.

    Turn types:
      - images: derived from <tool_response> ... </tool_response>
      - answer: derived from <answer> ... </answer>

    This intentionally ignores internal reasoning blocks.
    """
    text = str(pred_text or "").replace("<|endoftext|>", "")

    # Keep only user tool responses and assistant answers, in original order.
    pattern = re.compile(
        r"(<tool_response>.*?</tool_response>)|(<answer>.*?</answer>)",
        flags=re.DOTALL,
    )
    
    turns = []
    turn_idx = 1
    for m in pattern.finditer(text):
        tool_block = m.group(1)
        ans_block = m.group(2)

        # Anchor each event to the nearest preceding tool-response boundary.
        # This avoids index drift when `<answer>` appears inside reasoning text.
        event_start, event_end = m.span()
        prev_tool_close = text.rfind("</tool_response>", 0, event_start)
        if prev_tool_close == -1:
            turn_text_start = 0
        else:
            turn_text_start = prev_tool_close + len("</tool_response>")
        turn_text = text[turn_text_start:event_end]

        # Remove raw vision tokens so turn text is readable.
        clean_turn_text = re.sub(
            r"<\|(?:vision_start|vision_end|image_pad)\|>",
            "",
            turn_text,
        )
        clean_turn_text = re.sub(r"\s+", " ", clean_turn_text).strip()

        think_text = re.sub(r"<think>.*?</think>", "", clean_turn_text, flags=re.DOTALL).strip()
        tool_pattern = re.compile(
            r"<search>.*?</search>|<search_summary>.*?</search_summary>|<search_subtitle>.*?</search_subtitle>",
            flags=re.DOTALL,
        )
        tool_matches = list(tool_pattern.finditer(clean_turn_text))
        if tool_matches:
            # Keep only the last tool call block.
            last_tool_block = tool_matches[-1].group(0)
            tool_text = re.sub(r"</?(search|search_summary|search_subtitle)>", "", last_tool_block).strip()
        else:
            tool_text = ""
        
        if tool_block is not None:
            
            num_image_tokens = len(re.findall(r"<\|image_pad\|>", tool_block))
            num_vision_blocks = len(re.findall(r"<\|vision_start\|>", tool_block))

            turns.append(
                {
                    "turn_idx": turn_idx,
                    "turn_type": "images",
                    "num_image_tokens": num_image_tokens,
                    "num_vision_blocks": num_vision_blocks,
                    "think_text": think_text,
                    "tool_text": tool_text,
                    "clean_turn_text": clean_turn_text,
                }
            )
            turn_idx += 1
            continue

        if ans_block is not None:
            # Ignore tentative answers that are followed by another tool call.
            # These often appear inside reasoning traces, not final outputs.
            next_tool_response = text.find("<tool_response>", event_end)
            if next_tool_response != -1:
                continue

            answer_text = re.sub(r"</?answer>", "", ans_block).strip()
            turns.append(
                {
                    "turn_idx": turn_idx,
                    "turn_type": "answer",
                    "answer_text": answer_text,
                    "clean_turn_text": clean_turn_text,
                }
            )
            turn_idx += 1

    return turns


def majority_pred_text(row: dict, type: str):
    """Get pred text aligned to majority_vote for a single jsonl row."""
    preds = row.get("pred", []) or []
    parsed = row.get("parsed_pred", []) or []
    majority = str(row.get("majority_vote", ""))

    idx = 0
    for i, p in enumerate(parsed):
        if str(p) == majority:
            idx = i
            if type == "puls" and int(row.get("turns", 0)[i]) >= 6:
                break
            elif type == "em" and int(row.get("turns", 0)[i]) < 6:
                break
    return preds[idx] if idx < len(preds) else (preds[0] if preds else "")


# Demo on examples selected in previous cell
# (picked_vseek and picked_em_wrong come from Cell 5)

def print_turn_split(examples: list[dict]):
    print("\n" + "=" * 100)
    print("=" * 100)

    for ex in examples:
        if 'count' in ex['video_id'] or'anomaly' in ex['video_id'] or 'order' in ex['video_id'] or 'ego' in ex['video_id']:
            continue
        qid = ex["qid"]
        row = puls_map[qid] 
        raw_pred = majority_pred_text(row, type="puls")
        turns = split_pred_text_per_turn(raw_pred)
        print("-" * 100)
        print(ex['question'])
        print(ex['candidates'])
        print("." * 100)
        print("Puls")
        print(f"qid={qid} | video_id={ex['video_id']} | turns_extracted={len(turns)}")
        for t in turns:
            if t["turn_type"] == "images":
                print(
                    f"  - turn {t['turn_idx']}: images "
                    f"(vision_blocks={t['num_vision_blocks']}, image_tokens={t['num_image_tokens']})"
                    f"think: {t['think_text'][:280]}\n"
                    f"tool: {t['tool_text'][:280]}\n"
                    f"clean: {t['clean_turn_text']}\n"
                )
            else:
                print(
                    f"  - turn clean: {t['clean_turn_text']}\n {t['turn_idx']}: answer -> {t['answer_text']}\n"
                )
        
        
        row = em_map[qid] 
        raw_pred = majority_pred_text(row, type="em")
        turns = split_pred_text_per_turn(raw_pred)
        print("-" * 100)
        print("EM")
        print(f"qid={qid} | video_id={ex['video_id']} | turns_extracted={len(turns)}")
        for t in turns:
            if t["turn_type"] == "images":
                print(
                    f"  - turn {t['turn_idx']}: images "
                    f"(vision_blocks={t['num_vision_blocks']}, image_tokens={t['num_image_tokens']})"
                    f"think: {t['think_text'][:280]}\n"
                    f"clean: {t['clean_turn_text']}\n"
                    f"tool: {t['tool_text'][:280]}\n"
                    
                )
            else:
                print(
                    f"  - turn clean: {t['clean_turn_text']}\n {t['turn_idx']}: answer -> {t['answer_text']}\n"
                )
        print("-" * 100)
        print("-" * 100)

print(len(picked_vseek))
# Show split for the first two selected examples on both models
print_turn_split(all_vseek)

73

----------------------------------------------------------------------------------------------------

 This is a multiple choice question. You must choose the correct answer as a number or letter of the option. 
Question: In the black and white footage, there is a curly-haired woman wearing a black dress, holding a bag, standing on an empty street. What is she doing? 

['She is turning around and greeting someone', 'She is waiting for the bus by the roadside', 'She is walking on the street', 'She is looking through her bag']
....................................................................................................
Puls
qid=6 | video_id=hznvV2bBkX4 | turns_extracted=3
  - turn 1: images (vision_blocks=16, image_tokens=1120)think: Okay, let's try to figure out this question. The user is asking what the curly-haired woman in a black dress, holding a bag, standing on an empty street is doing. The options are 0 (turning around and greeting), 1 (waiting for the bus), 2 (walking